# Round 14: robustness checkpoint - a second fold partition for the Exp 36 estimate - S6E10

Every candidate since round 3 has been measured on the same outer partition (StratifiedKFold,
5 folds, seed 42), and the preferred candidate was chosen adaptively against that partition.
Exp 36's mean OOF is 0.959996 - 4e-6 below the strict 0.960 target. Before any more
same-partition tuning, this round asks a measurement question, not a tuning question: does the
Exp 36 estimate, and its +0.00014 increment over Exp 31, survive a change of outer partition?

One change: the outer partition seed (42 -> 123). Everything else - pipeline, features, TE
construction, ES split, configs, seeds - is identical to round 12. The incumbent, Exp 31, and
Exp 36 are re-scored under partition 123.

This round is NOT a candidate: the refit guard is not evaluated, no submission file is written
whatever the outcome, and the 0.960 target status is still judged on the established seed-42
partition - a good partition-123 score does not relabel the target as attained, and a bad one
does not by itself demote Exp 36; it tempers the claim.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
OUTER_SEED = 123  # the one change this round: a second outer fold partition
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CROSS_PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Class')]
# seed-42-partition references (different partition - descriptive comparison only, never paired)
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # incumbent, seed-42 partition
E31_FOLDS_R10 = [0.95998, 0.95873, 0.96056, 0.96000, 0.96000]       # counts + TE, seed-42 partition
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # preferred candidate, seed-42 partition
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100

In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def cross_name(pair):
    return ' x '.join(pair)

CROSS_COLS = [cross_name(p) for p in CROSS_PAIRS]

def add_crosses(df):
    out = df.copy()
    for pair in CROSS_PAIRS:
        out[cross_name(pair)] = out[pair[0]].astype(str) + ' x ' + out[pair[1]].astype(str)
    return out

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, te_drop=(),
             outer_seed=RANDOM_STATE, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)  # round 14: partition passed in
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
            if te_drop:  # raw crossed strings never enter the model - only their encodings
                X_fit = X_fit.drop(columns=list(te_drop))
                X_es = X_es.drop(columns=list(te_drop))
                X_va = X_va.drop(columns=list(te_drop))
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label, outer_seed=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; crossed columns: {CROSS_COLS}')

Full train: 699,635 rows, 21 raw features; crossed columns: ['Class x Type of Travel', 'Customer Type x Class']


## Reference: incumbent on partition 123

Same Exp 8 raw-LightGBM incumbent, new partition. The round-3 determinism check does not
apply here - different folds by construction - so the incumbent is a same-partition reference
for the two re-scores below, nothing more.

In [3]:
print('Reference: incumbent raw LightGBM (Exp 8 config), partition 123')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC-p123', outer_seed=OUTER_SEED)
print(f'  incumbent folds on partition 123: {[round(a, 5) for a in aucs_inc]}')
print(f'  (seed-42 partition reference was {INCUMBENT_FOLDS_R3} - different folds, no determinism claim)')
INC_FOLDS = aucs_inc

Reference: incumbent raw LightGBM (Exp 8 config), partition 123


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC-p123 fold 1: AUC 0.95845 (best_iteration=249)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC-p123 fold 2: AUC 0.95752 (best_iteration=392)


...

  INC-p123 fold 3: AUC 0.95892 (best_iteration=315)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC-p123 fold 4: AUC 0.95896 (best_iteration=270)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC-p123 fold 5: AUC 0.95870 (best_iteration=268)
  incumbent folds on partition 123: [0.95845, 0.95752, 0.95892, 0.95896, 0.9587]
  (seed-42 partition reference was [0.95851, 0.95751, 0.95907, 0.95841, 0.95885] - different folds, no determinism claim)


## Exp 31 re-score on partition 123

Counts + all-21 leak-free cross-fitted TE + stochastic 3-seed average (the round-10 scheme),
unchanged except the outer partition.

In [4]:
print('Exp 31 re-score: counts + all-21 TE + stochastic, seeds 42/43/44, partition 123')
oof_seeds31 = {}
for seed in [42, 43, 44]:
    o, a, it = oof_lgbm(features, target, f'E31-s{seed}-p123', seed=seed,
                        count_features=True, target_encode=True, outer_seed=OUTER_SEED, **STOCH)
    oof_seeds31[seed] = o
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof31 = np.mean([oof_seeds31[s] for s in [42, 43, 44]], axis=0)
aucs31 = fold_aucs(oof31, target, outer_seed=OUTER_SEED)
print(f'Exp 31 p123 fold AUCs: {[round(a, 5) for a in aucs31]}')
print(f'Exp 31 p123 mean OOF AUC: {np.mean(aucs31):.6f} +/- {np.std(aucs31):.5f} | pooled: {roc_auc_score(target, oof31):.6f}')
print(f'  (seed-42 partition: mean 0.959855 / pooled 0.959854)')

Exp 31 re-score: counts + all-21 TE + stochastic, seeds 42/43/44, partition 123


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s42-p123 fold 1: AUC 0.95986 (best_iteration=232)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s42-p123 fold 2: AUC 0.95856 (best_iteration=238)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s44-p123 fold 4: AUC 0.96004 (best_iteration=240)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s44-p123 fold 5: AUC 0.95991 (best_iteration=654)
  seed 44 mean AUC: 0.95965
Exp 31 p123 fold AUCs: [0.95988, 0.95853, 0.96049, 0.96006, 0.96013]


Exp 31 p123 mean OOF AUC: 0.959819 +/- 0.00067 | pooled: 0.959813
  (seed-42 partition: mean 0.959855 / pooled 0.959854)


## Exp 36 re-score on partition 123

The preferred candidate: counts + all-21 TE, 3 seeds x {63, 127} leaves, 6-model average.

In [5]:
print('Exp 36 re-score: counts + all-21 TE + stochastic, 3 seeds x {63, 127} leaves, partition 123')
LEAF_CONFIGS = [63, 127]
oof_runs36 = {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features, target, f'E36-l{leaves}-s{seed}-p123', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, outer_seed=OUTER_SEED, **STOCH)
        oof_runs36[(leaves, seed)] = o
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof36 = np.mean(list(oof_runs36.values()), axis=0)
aucs36 = fold_aucs(oof36, target, outer_seed=OUTER_SEED)
print(f'Exp 36 p123 fold AUCs: {[round(a, 5) for a in aucs36]}')
print(f'Exp 36 p123 mean OOF AUC: {np.mean(aucs36):.6f} +/- {np.std(aucs36):.5f} | pooled: {roc_auc_score(target, oof36):.6f}')
print(f'  (seed-42 partition: mean 0.959996 / pooled 0.959992)')

Exp 36 re-score: counts + all-21 TE + stochastic, 3 seeds x {63, 127} leaves, partition 123


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E36-l63-s42-p123 fold 1: AUC 0.95986 (best_iteration=232)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E36-l63-s42-p123 fold 2: AUC 0.95856 (best_iteration=238)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E36-l127-s44-p123 fold 4: AUC 0.95993 (best_iteration=174)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E36-l127-s44-p123 fold 5: AUC 0.96016 (best_iteration=234)
  leaves=127 seed 44 mean AUC: 0.95976
Exp 36 p123 fold AUCs: [0.96005, 0.9587, 0.96065, 0.96016, 0.96027]


Exp 36 p123 mean OOF AUC: 0.959967 +/- 0.00067 | pooled: 0.959961
  (seed-42 partition: mean 0.959996 / pooled 0.959992)


## Summary: does the estimate survive the partition change?

Descriptive only - the seed-42 and seed-123 partitions are different folds, so no paired
verdict across partitions. The questions: (1) is Exp 36's new-partition mean within roughly
2e-4 of 0.959996? (2) does Exp 36 still score above Exp 31 on the new partition, and by
roughly how much vs the +0.00014 measured on seed 42?

In [6]:
m_inc, m_31, m_36 = float(np.mean(aucs_inc)), float(np.mean(aucs31)), float(np.mean(aucs36))
print(f'partition 123 means: incumbent {m_inc:.6f} | Exp 31 {m_31:.6f} | Exp 36 {m_36:.6f}')
print(f'Exp 36 minus Exp 31 on partition 123: {m_36 - m_31:+.6f}  (seed-42 partition: +0.000141)')
print(f'Exp 36 minus incumbent on partition 123: {m_36 - m_inc:+.6f}  (seed-42 partition: +0.001526)')
print(f'Exp 36 p123 mean distance from strict 0.960: {0.96 - m_36:+.6f}')
print()
if abs(m_36 - 0.959996) <= 2e-4 and m_36 > m_31:
    print('READING: estimate robust to the partition change in this test (descriptive, not proof).')
elif m_36 <= m_31:
    print('READING: Exp 36 does NOT beat Exp 31 on the new partition - the round-12 increment looks partition-dependent; claims temper.')
else:
    print('READING: estimate moved more than 2e-4 under the partition change - treat 0.959996 as partition-optimistic; claims temper.')
print()
print('CHECKPOINT BOUNDARY: this round is not a candidate. The refit guard was not evaluated, '
      'no submission file was written, submission_round12.csv (Exp 36) is untouched, and the '
      'strict 0.960 target status is unchanged - it is judged on the established seed-42 partition '
      '(mean 0.959996, unmet), not on this checkpoint partition.')

partition 123 means: incumbent 0.958510 | Exp 31 0.959819 | Exp 36 0.959967
Exp 36 minus Exp 31 on partition 123: +0.000147  (seed-42 partition: +0.000141)
Exp 36 minus incumbent on partition 123: +0.001457  (seed-42 partition: +0.001526)
Exp 36 p123 mean distance from strict 0.960: +0.000033

READING: estimate robust to the partition change in this test (descriptive, not proof).

CHECKPOINT BOUNDARY: this round is not a candidate. The refit guard was not evaluated, no submission file was written, submission_round12.csv (Exp 36) is untouched, and the strict 0.960 target status is unchanged - it is judged on the established seed-42 partition (mean 0.959996, unmet), not on this checkpoint partition.


In [7]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round14_oof.npz',
         incumbent_p123=oof_inc, exp31_p123=oof31, exp36_p123=oof36, target=target.to_numpy())
meta = {
    'round': 14,
    'kind': 'robustness checkpoint - not a candidate, no refit',
    'outer_seed': OUTER_SEED,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'fold_aucs': {'incumbent_p123': aucs_inc, 'exp31_p123': aucs31, 'exp36_p123': aucs36},
    'seed42_references': {'incumbent_r3': INCUMBENT_FOLDS_R3, 'exp31_r10': E31_FOLDS_R10, 'exp36_r12': E36_FOLDS_R12},
}
(cache_dir / 'round14_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round14_oof.npz + round14_oof.json')

cached oof_cache/round14_oof.npz + round14_oof.json


## What happens next

- If the estimate is robust: the plateau is real on this pipeline - the honest options are an
  informed bounded batch on existing data (levers are thinning: forests measured below the
  booster, blends failed on correlated pairs) or the external-data question to Andrea.
- If the estimate is partition-optimistic: all seed-42 comparisons soften, and the campaign
  re-baselines expectations before any further tuning claims.
- Either way Exp 36 and submission_round12.csv stand; this checkpoint changed the evidence,
  not the candidate.